# SDXL LoRA Eğitimi — Kohya SS

Karakter veya stil LoRA eğitimi. 96GB VRAM ile bf16 full precision, hızlı eğitim.

## Dataset Hazırlık
- Karakter: 20-50 görsel
- Stil: 30-100 görsel
- Çözünürlük: 1024x1024
- Her görsele caption (.txt dosyası, aynı isimle)

## Kullanım
A: Kurulum → B: Dataset yükle → C: Config → D: Eğitim → E: LoRA indir

---
# A) Kurulum

In [ ]:
import os
import torch

if not torch.cuda.is_available():
    raise RuntimeError('GPU bulunamadı!')
gpu_name = torch.cuda.get_device_name(0)
gpu_mem = torch.cuda.get_device_properties(0).total_memory / 1024**3
print(f'\u2705 GPU: {gpu_name} ({gpu_mem:.1f} GB)')

os.environ['HF_HUB_DISABLE_TELEMETRY'] = '1'
try:
    from google.colab import userdata
    os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')
    print('\u2705 HF_TOKEN')
except Exception:
    print('\u26a0\ufe0f HF_TOKEN yok')

# Kohya sd-scripts kurulumu
KOHYA_DIR = '/content/sd-scripts'
if not os.path.exists(KOHYA_DIR):
    print('\U0001f4e6 Kohya sd-scripts indiriliyor...')
    !git clone --depth 1 https://github.com/kohya-ss/sd-scripts.git {KOHYA_DIR}
    !pip install -q -r {KOHYA_DIR}/requirements.txt
    !pip install -q bitsandbytes prodigyopt lion-pytorch
else:
    print('\u2705 Kohya sd-scripts mevcut')

# SDXL base model indir
MODEL_DIR = '/content/models'
SDXL_PATH = f'{MODEL_DIR}/sd_xl_base_1.0.safetensors'
os.makedirs(MODEL_DIR, exist_ok=True)
if not os.path.exists(SDXL_PATH):
    print('\U0001f4e5 SDXL 1.0 base indiriliyor...')
    from huggingface_hub import hf_hub_download
    path = hf_hub_download('stabilityai/stable-diffusion-xl-base-1.0', 'sd_xl_base_1.0.safetensors', local_dir=MODEL_DIR)
    print('\u2705 SDXL indirildi')
else:
    print('\u2705 SDXL mevcut')

print('\u2705 Kurulum tamam')

---
# B) Dataset Yükle

Zip yükle:
```
dataset/
├── image1.jpg
├── image1.txt
├── image2.png
├── image2.txt
└── ...
```

In [ ]:
import zipfile
from google.colab import files

DATASET_DIR = '/content/dataset'

print('Dataset zip dosyasını yükle:')
uploaded = files.upload()
zip_name = list(uploaded.keys())[0]

os.makedirs(DATASET_DIR, exist_ok=True)
with zipfile.ZipFile(zip_name, 'r') as z:
    z.extractall(DATASET_DIR)

# İç içe klasör varsa düzelt
subdirs = [d for d in os.listdir(DATASET_DIR) if os.path.isdir(f'{DATASET_DIR}/{d}')]
if len(subdirs) == 1 and not any(f.endswith(('.jpg', '.png', '.webp')) for f in os.listdir(DATASET_DIR)):
    inner = f'{DATASET_DIR}/{subdirs[0]}'
    for f in os.listdir(inner):
        os.rename(f'{inner}/{f}', f'{DATASET_DIR}/{f}')
    os.rmdir(inner)

images = [f for f in os.listdir(DATASET_DIR) if f.lower().endswith(('.jpg', '.jpeg', '.png', '.webp'))]
captions = [f for f in os.listdir(DATASET_DIR) if f.endswith('.txt')]
print(f'\u2705 {len(images)} görsel, {len(captions)} caption')
if len(images) != len(captions):
    print('\u26a0\ufe0f Her görselin .txt caption\'ı olmalı!')

---
# C) Eğitim Config

In [ ]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  EĞİTİM CONFIG                                            ║
# ╚══════════════════════════════════════════════════════════════╝
LORA_NAME       = 'my_sdxl_lora'
TRIGGER_WORD    = 'ohk_char'
NETWORK_RANK    = 32             # 32-64 SDXL için
NETWORK_ALPHA   = 16             # rank/2 önerilen
LEARNING_RATE   = 1e-4           # UNet LR
TE_LR           = 5e-5           # Text Encoder LR (0 = kapalı)
BATCH_SIZE      = 4              # 96GB VRAM ile 4-8
EPOCHS          = 15             # 10-20
RESOLUTION      = 1024           # SDXL native
OPTIMIZER       = 'prodigy'      # 'prodigy' | 'adamw8bit' | 'lion'
SCHEDULER       = 'cosine_with_restarts'
SAVE_EVERY      = 5              # Her N epoch'ta kaydet
CAPTION_DROPOUT = 0.05           # %5 caption dropout

OUTPUT_DIR = f'/content/output/{LORA_NAME}'
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Prodigy kullanırken LR otomatik ayarlanır
if OPTIMIZER == 'prodigy':
    LEARNING_RATE = 1.0
    TE_LR = 1.0
    print('\u2139\ufe0f Prodigy optimizer — LR otomatik ayarlanacak')

print(f'\u2705 Config:')
print(f'   Name:       {LORA_NAME}')
print(f'   Trigger:    {TRIGGER_WORD}')
print(f'   Rank/Alpha: {NETWORK_RANK}/{NETWORK_ALPHA}')
print(f'   Optimizer:  {OPTIMIZER}')
print(f'   Batch:      {BATCH_SIZE}')
print(f'   Epochs:     {EPOCHS}')
print(f'   Dataset:    {len(images)} images')

---
# D) Eğitim Başlat

In [ ]:
import subprocess
import time

# Kohya dataset config (TOML)
TOML_PATH = '/content/dataset.toml'
toml_content = f"""[general]
shuffle_caption = true
caption_extension = '.txt'
keep_tokens = 1

[[datasets]]
resolution = {RESOLUTION}
batch_size = {BATCH_SIZE}
enable_bucket = true

  [[datasets.subsets]]
  image_dir = '{DATASET_DIR}'
  num_repeats = 1
  caption_dropout_rate = {CAPTION_DROPOUT}
"""

with open(TOML_PATH, 'w') as f:
    f.write(toml_content)

# Eğitim komutu
cmd = [
    'python', 'sdxl_train_network.py',
    '--pretrained_model_name_or_path', SDXL_PATH,
    '--dataset_config', TOML_PATH,
    '--output_dir', OUTPUT_DIR,
    '--output_name', LORA_NAME,
    '--network_module', 'networks.lora',
    '--network_dim', str(NETWORK_RANK),
    '--network_alpha', str(NETWORK_ALPHA),
    '--learning_rate', str(LEARNING_RATE),
    '--text_encoder_lr', str(TE_LR),
    '--max_train_epochs', str(EPOCHS),
    '--save_every_n_epochs', str(SAVE_EVERY),
    '--optimizer_type', OPTIMIZER,
    '--lr_scheduler', SCHEDULER,
    '--mixed_precision', 'bf16',
    '--save_precision', 'fp16',
    '--cache_latents',
    '--cache_latents_to_disk',
    '--no_half_vae',
    '--seed', '42',
]

# Prodigy-specific args
if OPTIMIZER == 'prodigy':
    cmd.extend(['--optimizer_args', 'weight_decay=0.01', 'decouple=True', 'use_bias_correction=True'])

print(f'\U0001f680 Eğitim başlıyor: {LORA_NAME}')
print(f'   {EPOCHS} epoch × {len(images)} images × batch {BATCH_SIZE}\n')

t0 = time.time()
result = subprocess.run(cmd, cwd=KOHYA_DIR, capture_output=False)

elapsed = (time.time() - t0) / 60
if result.returncode == 0:
    print(f'\n\u2705 Eğitim tamamlandı! ({elapsed:.1f} dakika)')
else:
    print(f'\n\u274c Eğitim başarısız (exit: {result.returncode})')

---
# E) LoRA İndir

In [ ]:
import glob
from google.colab import files

lora_files = sorted(glob.glob(f'{OUTPUT_DIR}/*.safetensors'))

if lora_files:
    print(f'\u2705 LoRA dosyaları ({len(lora_files)}):')
    for f in lora_files:
        size_mb = os.path.getsize(f) / 1024**2
        print(f'   {os.path.basename(f)} ({size_mb:.1f} MB)')

    final = lora_files[-1]
    print(f'\n\U0001f4e6 İndiriliyor: {os.path.basename(final)}')
    files.download(final)
    print(f'\n\U0001f4cb Kullanım:')
    print(f'   ComfyUI/models/loras/ klasörüne koy')
    print(f'   Prompt\'ta "{TRIGGER_WORD}" kullan')
else:
    print('\u274c LoRA dosyası bulunamadı!')